In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 1 ── INSTALL PACKAGES
# Run this cell once, then go to Runtime > Restart Runtime before Cell 2
# ═══════════════════════════════════════════════════════════════════════════════

# !nvidia-smi
# !pip install -q openai-whisper
# !pip install -q sentence-transformers
# !pip install -q faiss-cpu
# !pip install -q ffmpeg-python
# !pip install -q torch torchvision torchaudio
# !pip install -q transformers timm pillow opencv-python sentencepiece
# !pip install -q rouge-score bert-score
# !apt-get update -qq
# !apt-get install -y ffmpeg -qq
# print("✅ All packages installed. Go to Runtime > Restart Runtime now.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 2 ── IMPORTS, FOLDER SETUP, 8 CONFIGS
# ═══════════════════════════════════════════════════════════════════════════════

import os, json, re, cv2, subprocess, time, datetime, base64, warnings
import torch
import faiss
import numpy as np
import whisper
from sentence_transformers import SentenceTransformer
from transformers import (
    BlipProcessor, BlipForConditionalGeneration, pipeline
)
from PIL import Image
from rouge_score import rouge_scorer
from IPython.display import display, HTML, clear_output
from google.colab import files

warnings.filterwarnings("ignore")

# ── Folders (same as original code) ──────────────────────────────────────────
for d in ["video", "frames", "transcripts", "embeddings", "clips", "results"]:
    os.makedirs(d, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Setup complete  |  Device: {DEVICE}")

# ══════════════════════════════════════════════════════════════════════════════
#  8 NAMED PARAMETER CONFIGURATIONS
#
#  Each config has a clear hypothesis — what aspect it is testing.
#  Parameters not listed for a model use the same default as Config_1.
#  Models themselves do not change — only their generation parameters change,
#  except where model_size / model_variant is explicitly listed.
# ══════════════════════════════════════════════════════════════════════════════

CONFIGS = {

    # ── Config 1: Baseline — your original defaults ───────────────────────────
    "Config_1_Baseline": {
        "hypothesis": "Original defaults — establishes the performance baseline",
        # Whisper
        "whisper_model_size":    "base",
        "whisper_temperature":   0.0,
        "whisper_beam_size":     5,
        # BLIP
        "blip_model_size":       "Salesforce/blip-image-captioning-base",
        "blip_num_beams":        3,
        "blip_min_new_tokens":   10,
        "blip_max_new_tokens":   30,
        # SentenceTransformer
        "embed_model":           "all-MiniLM-L6-v2",
        # FAISS
        "frame_interval_s":      2,
        "top_k":                 2,
        # FLAN-T5
        "flan_temperature":      1.0,
        "flan_num_beams":        1,
        "flan_repetition_penalty": 1.0,
        "flan_length_penalty":   1.0,
        "flan_max_length":       64,
    },

    # ── Config 2: Better ASR — improved transcription quality ─────────────────
    "Config_2_BetterASR": {
        "hypothesis": "Whisper-small with higher beam search improves transcript accuracy",
        "whisper_model_size":    "small",      # ← changed
        "whisper_temperature":   0.0,
        "whisper_beam_size":     5,            # ← changed
        "blip_model_size":       "Salesforce/blip-image-captioning-base",
        "blip_num_beams":        3,
        "blip_min_new_tokens":   10,
        "blip_max_new_tokens":   30,
        "embed_model":           "all-MiniLM-L6-v2",
        "frame_interval_s":      2,
        "top_k":                 2,
        "flan_temperature":      1.0,
        "flan_num_beams":        1,
        "flan_repetition_penalty": 1.0,
        "flan_length_penalty":   1.0,
        "flan_max_length":       64,
    },

    # ── Config 3: Deterministic ASR — zero randomness in transcription ────────
    "Config_3_DeterministicASR": {
        "hypothesis": "Temperature=0 with beam_size=5 gives most consistent transcription",
        "whisper_model_size":    "small",      # ← changed
        "whisper_temperature":   0.0,          # ← fully deterministic
        "whisper_beam_size":     5,            # ← changed
        "blip_model_size":       "Salesforce/blip-image-captioning-base",
        "blip_num_beams":        3,
        "blip_min_new_tokens":   10,
        "blip_max_new_tokens":   30,
        "embed_model":           "all-MiniLM-L6-v2",
        "frame_interval_s":      2,
        "top_k":                 2,
        "flan_temperature":      1.0,
        "flan_num_beams":        1,
        "flan_repetition_penalty": 1.0,
        "flan_length_penalty":   1.0,
        "flan_max_length":       64,
    },

    # ── Config 4: Richer Visual Captions — BLIP-large with stronger decoding ──
    "Config_4_RicherCaptions": {
        "hypothesis": "BLIP-large with longer min tokens produces richer visual evidence",
        "whisper_model_size":    "base",
        "whisper_temperature":   0.0,
        "whisper_beam_size":     5,
        "blip_model_size":       "Salesforce/blip-image-captioning-large",  # ← changed
        "blip_num_beams":        5,            # ← changed
        "blip_min_new_tokens":   20,           # ← changed: forces longer captions
        "blip_max_new_tokens":   50,           # ← changed
        "embed_model":           "all-MiniLM-L6-v2",
        "frame_interval_s":      2,
        "top_k":                 2,
        "flan_temperature":      1.0,
        "flan_num_beams":        1,
        "flan_repetition_penalty": 1.0,
        "flan_length_penalty":   1.0,
        "flan_max_length":       64,
    },

    # ── Config 5: Better Retrieval — stronger embedding model ─────────────────
    "Config_5_BetterRetrieval": {
        "hypothesis": "all-mpnet-base-v2 embedding improves semantic retrieval accuracy",
        "whisper_model_size":    "base",
        "whisper_temperature":   0.0,
        "whisper_beam_size":     5,
        "blip_model_size":       "Salesforce/blip-image-captioning-base",
        "blip_num_beams":        3,
        "blip_min_new_tokens":   10,
        "blip_max_new_tokens":   30,
        "embed_model":           "all-mpnet-base-v2",   # ← changed
        "frame_interval_s":      2,
        "top_k":                 3,                     # ← changed
        "flan_temperature":      1.0,
        "flan_num_beams":        1,
        "flan_repetition_penalty": 1.0,
        "flan_length_penalty":   1.0,
        "flan_max_length":       64,
    },

    # ── Config 6: Higher Quality Answers — beam search + anti-repetition ──────
    "Config_6_BetterAnswers": {
        "hypothesis": "FLAN beam search + repetition penalty improves answer quality",
        "whisper_model_size":    "base",
        "whisper_temperature":   0.0,
        "whisper_beam_size":     5,
        "blip_model_size":       "Salesforce/blip-image-captioning-base",
        "blip_num_beams":        3,
        "blip_min_new_tokens":   10,
        "blip_max_new_tokens":   30,
        "embed_model":           "all-MiniLM-L6-v2",
        "frame_interval_s":      2,
        "top_k":                 2,
        "flan_temperature":      1.0,
        "flan_num_beams":        4,            # ← changed: beam search in answer gen
        "flan_repetition_penalty": 1.3,        # ← changed: reduces repetition
        "flan_length_penalty":   1.0,
        "flan_max_length":       64,
    },

    # ── Config 7: Verbose Answers — longer, more detailed responses ───────────
    "Config_7_VerboseAnswers": {
        "hypothesis": "Higher length_penalty + beam search produces more detailed answers",
        "whisper_model_size":    "base",
        "whisper_temperature":   0.0,
        "whisper_beam_size":     5,
        "blip_model_size":       "Salesforce/blip-image-captioning-base",
        "blip_num_beams":        3,
        "blip_min_new_tokens":   10,
        "blip_max_new_tokens":   30,
        "embed_model":           "all-MiniLM-L6-v2",
        "frame_interval_s":      2,
        "top_k":                 3,            # ← changed
        "flan_temperature":      1.0,
        "flan_num_beams":        4,            # ← changed
        "flan_repetition_penalty": 1.3,        # ← changed
        "flan_length_penalty":   1.5,          # ← changed: encourages longer answers
        "flan_max_length":       128,          # ← changed
    },

    # ── Config 8: Best Combined — all strongest settings together ─────────────
    "Config_8_BestCombined": {
        "hypothesis": "Combining best ASR + retrieval + answer settings for maximum quality",
        "whisper_model_size":    "small",      # from Config_2
        "whisper_temperature":   0.0,
        "whisper_beam_size":     5,
        "blip_model_size":       "Salesforce/blip-image-captioning-large",  # from Config_4
        "blip_num_beams":        5,
        "blip_min_new_tokens":   20,
        "blip_max_new_tokens":   50,
        "embed_model":           "all-mpnet-base-v2",   # from Config_5
        "frame_interval_s":      2,
        "top_k":                 3,
        "flan_temperature":      1.0,
        "flan_num_beams":        4,            # from Config_6
        "flan_repetition_penalty": 1.3,
        "flan_length_penalty":   1.5,          # from Config_7
        "flan_max_length":       128,
    },
}

print(f"✅ {len(CONFIGS)} configurations defined")
print("\nConfig summary:")
for name, cfg in CONFIGS.items():
    print(f"  {name}: {cfg['hypothesis']}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 3 ── CORE PIPELINE FUNCTIONS
# Based exactly on your original Colab code structure.
# Each function now accepts a cfg dict for its parameters.
# ═══════════════════════════════════════════════════════════════════════════════

# ── Model cache — avoids reloading the same model repeatedly ──────────────────
_whisper_cache = {}
_embed_cache   = {}
_blip_cache    = {}
_flan_cache    = {}

def _get_whisper(model_size):
    if model_size not in _whisper_cache:
        print(f"      Loading Whisper '{model_size}'...")
        _whisper_cache[model_size] = whisper.load_model(model_size)
    return _whisper_cache[model_size]

def _get_embed(model_name):
    if model_name not in _embed_cache:
        print(f"      Loading embeddings '{model_name}'...")
        _embed_cache[model_name] = SentenceTransformer(model_name)
    return _embed_cache[model_name]

def _get_blip(model_size):
    if model_size not in _blip_cache:
        print(f"      Loading BLIP '{model_size}'...")
        proc  = BlipProcessor.from_pretrained(model_size)
        model = BlipForConditionalGeneration.from_pretrained(model_size).to(DEVICE)
        _blip_cache[model_size] = (proc, model)
    return _blip_cache[model_size]

def _get_flan():
    if "flan" not in _flan_cache:
        print("      Loading FLAN-T5...")
        _flan_cache["flan"] = pipeline(
            "text2text-generation",
            model="google/flan-t5-base",
            device=0 if DEVICE == "cuda" else -1
        )
    return _flan_cache["flan"]


# ── VTT parser ────────────────────────────────────────────────────────────────

def parse_vtt(vtt_text):
    """
    Parse a WebVTT file into a list of transcript segments.
    Returns same format as Whisper: [{"text":..., "start":..., "end":...}]
    """
    segments = []
    # Remove BOM and header
    vtt_text = vtt_text.replace("\ufeff", "")
    blocks   = re.split(r"\n\n+", vtt_text.strip())

    for block in blocks:
        lines = block.strip().splitlines()
        # Find the timestamp line
        ts_line  = None
        txt_lines = []
        for i, line in enumerate(lines):
            if "-->" in line:
                ts_line = line
                txt_lines = lines[i+1:]
                break
        if ts_line is None or not txt_lines:
            continue

        # Parse timestamps  00:00:01.000 --> 00:00:04.000
        ts_match = re.match(
            r"(\d+):(\d+):(\d+[\.,]\d+)\s*-->\s*(\d+):(\d+):(\d+[\.,]\d+)",
            ts_line.strip()
        )
        if not ts_match:
            continue

        def to_seconds(h, m, s):
            return int(h)*3600 + int(m)*60 + float(s.replace(",","."))

        start = to_seconds(*ts_match.group(1,2,3))
        end   = to_seconds(*ts_match.group(4,5,6))

        # Clean text: remove HTML tags, speaker labels
        text = " ".join(txt_lines)
        text = re.sub(r"<[^>]+>", "", text)
        text = re.sub(r"^\s*[\w\s]+:\s*", "", text)
        text = text.strip()

        if text:
            segments.append({"text": text, "start": start, "end": end})

    return segments


# ── Step 1: Transcription (Whisper OR VTT) ────────────────────────────────────

def step_transcribe(video_path, cfg, vtt_segments=None):
    """
    If VTT segments are provided, use them directly (skips Whisper).
    Otherwise run Whisper with the config parameters.
    Returns list of {"text", "start", "end"} dicts.
    """
    if vtt_segments:
        print(f"   [1] Using VTT transcript → {len(vtt_segments)} segments")
        return vtt_segments

    print(f"   [1] Whisper transcription "
          f"(model={cfg['whisper_model_size']}, "
          f"temp={cfg['whisper_temperature']}, "
          f"beams={cfg['whisper_beam_size']})...")

    model  = _get_whisper(cfg["whisper_model_size"])
    result = model.transcribe(
        video_path,
        word_timestamps  = True,
        temperature      = cfg["whisper_temperature"],
        beam_size        = cfg["whisper_beam_size"],
    )
    segments = [
        {"text": s["text"].strip(), "start": s["start"], "end": s["end"]}
        for s in result["segments"]
    ]
    print(f"      → {len(segments)} segments")
    return segments


# ── Step 2: Text embeddings + FAISS index ─────────────────────────────────────

def step_text_index(segments, cfg):
    """
    Encode transcript segments with SentenceTransformer.
    Build FAISS IndexFlatL2. Exactly as in original code.
    """
    print(f"   [2] Text embeddings (model={cfg['embed_model']})...")
    embed_model  = _get_embed(cfg["embed_model"])
    texts        = [s["text"] for s in segments]
    text_embeddings = embed_model.encode(texts, convert_to_numpy=True)

    dim         = text_embeddings.shape[1]
    text_index  = faiss.IndexFlatL2(dim)
    text_index.add(text_embeddings)

    print(f"      → {text_index.ntotal} vectors | dim={dim}")
    return embed_model, text_embeddings, text_index, dim


# ── Step 3: Frame extraction ──────────────────────────────────────────────────

def step_extract_frames(video_path, cfg, tag):
    """
    Extract frames at cfg['frame_interval_s'] intervals.
    Saves to frames/<tag>/. Exactly as in original code.
    """
    print(f"   [3] Frame extraction (every {cfg['frame_interval_s']}s)...")
    frame_dir = f"frames/{tag}"
    os.makedirs(frame_dir, exist_ok=True)

    cap            = cv2.VideoCapture(video_path)
    fps            = cap.get(cv2.CAP_PROP_FPS) or 25
    frame_interval = int(fps * cfg["frame_interval_s"])
    frame_id, saved = 0, 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if frame_id % frame_interval == 0:
            timestamp = frame_id / fps
            cv2.imwrite(f"{frame_dir}/frame_{timestamp:.2f}.jpg", frame)
            saved += 1
        frame_id += 1
    cap.release()

    print(f"      → {saved} frames saved")
    return frame_dir, fps


# ── Step 4: BLIP visual captioning ───────────────────────────────────────────

def step_visual_captions(frame_dir, cfg):
    """
    Generate captions for each frame using BLIP.
    Returns list of {"text", "start", "end"} dicts.
    Exactly as in original code, now parameterised.
    """
    print(f"   [4] BLIP captions "
          f"(model={cfg['blip_model_size'].split('/')[-1]}, "
          f"beams={cfg['blip_num_beams']}, "
          f"min_tokens={cfg['blip_min_new_tokens']})...")

    processor, blip_model = _get_blip(cfg["blip_model_size"])
    visual_segments = []

    for img_file in sorted(os.listdir(frame_dir)):
        if not img_file.endswith(".jpg"):
            continue
        image   = Image.open(f"{frame_dir}/{img_file}").convert("RGB")
        inputs  = processor(image, return_tensors="pt").to(DEVICE)
        out     = blip_model.generate(
            **inputs,
            num_beams      = cfg["blip_num_beams"],
            min_new_tokens = cfg["blip_min_new_tokens"],
            max_new_tokens = cfg["blip_max_new_tokens"],
        )
        caption   = processor.decode(out[0], skip_special_tokens=True)
        timestamp = float(img_file.split("_")[1].replace(".jpg", ""))
        visual_segments.append({
            "text":  caption,
            "start": timestamp,
            "end":   timestamp + cfg["frame_interval_s"],
        })

    print(f"      → {len(visual_segments)} captions")
    return visual_segments


# ── Step 5: Visual embeddings + FAISS index ───────────────────────────────────

def step_visual_index(visual_segments, embed_model, dim):
    """
    Encode visual captions and build FAISS index.
    Exactly as in original code.
    """
    print(f"   [5] Visual FAISS index...")
    visual_texts      = [v["text"] for v in visual_segments]
    visual_embeddings = embed_model.encode(visual_texts, convert_to_numpy=True)

    visual_index = faiss.IndexFlatL2(dim)
    visual_index.add(visual_embeddings)

    print(f"      → {visual_index.ntotal} vectors")
    return visual_embeddings, visual_index


# ── Step 6: Query → answer + clip ─────────────────────────────────────────────

def step_answer_query(query, segments, text_index,
                      visual_segments, visual_index,
                      embed_model, cfg, video_path, tag):
    """
    Retrieve top-k text + visual segments, generate answer with FLAN-T5,
    extract supporting clip. Exactly as in original code, parameterised.
    """
    query_embedding = embed_model.encode([query], convert_to_numpy=True)

    # Text retrieval
    D_t, I_t = text_index.search(query_embedding, k=cfg["top_k"])
    text_context = " ".join(segments[i]["text"] for i in I_t[0])

    # Visual retrieval
    D_v, I_v = visual_index.search(query_embedding, k=cfg["top_k"])
    visual_context = " ".join(visual_segments[i]["text"] for i in I_v[0])

    combined_context = text_context + " " + visual_context

    # FLAN-T5 answer generation
    prompt = (
        f"Answer ONLY using the context below. Be concise.\n"
        f"Context: {combined_context}\n"
        f"Question: {query}"
    )
    flan = _get_flan()
    answer = flan(
        prompt,
        max_length         = cfg["flan_max_length"],
        num_beams          = cfg["flan_num_beams"],
        temperature        = cfg["flan_temperature"],
        repetition_penalty = cfg["flan_repetition_penalty"],
        length_penalty     = cfg["flan_length_penalty"],
        do_sample          = False,
    )[0]["generated_text"]

    # FFmpeg clip extraction — best matching segment
    best_segment = segments[I_t[0][0]]
    clip_start   = max(0, best_segment["start"] - 0.9)
    clip_end     = best_segment["end"] + 0.5
    clip_path    = f"clips/{tag}.mp4"

    subprocess.run([
        "ffmpeg", "-y", "-i", video_path,
        "-ss", str(clip_start),
        "-to", str(clip_end),
        clip_path
    ], capture_output=True)

    return {
        "answer":          answer,
        "clip_path":       clip_path,
        "clip_start":      clip_start,
        "clip_end":        clip_end,
        "clip_dur_s":      round(clip_end - clip_start, 2),
        "text_context":    text_context,
        "visual_context":  visual_context,
        "best_seg_start":  best_segment["start"],
        "best_seg_end":    best_segment["end"],
        "retrieval_dist":  float(D_t[0][0]),
    }


# ── Automated metric helpers ──────────────────────────────────────────────────

_rouge_scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"], use_stemmer=True
)

def auto_ar(generated, reference):
    """
    Automated AR proxy using ROUGE-1 F1.
    Maps ROUGE-1 score to 0/1/2 scale matching your paper.
    """
    if not reference:
        return None, None
    scores  = _rouge_scorer.score(reference, generated)
    rouge1  = round(scores["rouge1"].fmeasure, 4)
    # Map to 0-2 scale: <0.3=0, 0.3-0.6=1, >=0.6=2
    ar_auto = 2 if rouge1 >= 0.6 else (1 if rouge1 >= 0.3 else 0)
    return rouge1, ar_auto

def auto_egs(generated, text_context):
    """
    Automated EGS: fraction of content words in generated answer
    that appear in the retrieved context.
    Maps to 0/1/2 scale: <0.4=0, 0.4-0.7=1, >=0.7=2
    """
    stop = {"a","an","the","is","are","was","were","be","to","of","in",
            "for","on","with","it","its","this","that","and","or","but"}
    def words(t):
        return [w for w in re.sub(r"[^\w\s]","",t.lower()).split()
                if w not in stop and len(w) > 2]
    gen_w = words(generated)
    ctx_w = set(words(text_context))
    if not gen_w:
        return 0.0, 0
    rate    = round(sum(1 for w in gen_w if w in ctx_w) / len(gen_w), 4)
    egs_auto = 2 if rate >= 0.7 else (1 if rate >= 0.4 else 0)
    return rate, egs_auto

def auto_tla(clip_start, clip_end, vtt_segments, query_text):
    """
    Automated TLA using IoU between the extracted clip window
    and the VTT segment most relevant to the query.
    Only meaningful when VTT is provided.
    """
    if not vtt_segments:
        return None, None
    # Find VTT segment whose text best overlaps with query words
    stop = {"a","an","the","is","are","was","were","what","how","why",
            "which","does","do","did","can","could"}
    q_words = set(w.lower() for w in query_text.split() if w.lower() not in stop)
    best_overlap, best_seg = 0, None
    for seg in vtt_segments:
        seg_words = set(seg["text"].lower().split())
        overlap   = len(q_words & seg_words)
        if overlap > best_overlap:
            best_overlap, best_seg = overlap, seg
    if best_seg is None:
        return None, None
    ref_s, ref_e = best_seg["start"], best_seg["end"]
    inter = max(0, min(clip_end, ref_e) - max(clip_start, ref_s))
    union = max(1e-6, max(clip_end, ref_e) - min(clip_start, ref_s))
    iou   = round(inter / union, 4)
    tla_auto = 1 if iou >= 0.5 else 0
    return iou, tla_auto


print("✅ Pipeline functions ready")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 4 ── MAIN MULTI-VIDEO LOOP
#
# For each video:
#   1. Upload MP4 + VTT
#   2. Type your queries
#   3. All 8 configs run automatically
#   4. You see all answers (B2 style) and score them
#   5. Prompted to continue or finish
# ═══════════════════════════════════════════════════════════════════════════════

# Master results store — accumulates across all videos
MASTER = {}

SESSION_START = datetime.datetime.now()
video_count   = 0

while True:

    video_count += 1
    print(f"\n{'═'*65}")
    print(f"  VIDEO {video_count}")
    print(f"{'═'*65}")

    # ── Upload MP4 ────────────────────────────────────────────────────────────
    print(f"\n📁  Upload your MP4 file (video {video_count}):")
    uploaded_mp4 = files.upload()
    mp4_name     = list(uploaded_mp4.keys())[0]
    video_path   = f"video/input_v{video_count}.mp4"
    os.rename(mp4_name, video_path)
    video_label  = os.path.splitext(mp4_name)[0][:30]
    print(f"  ✅ Video saved: {video_path}")

    # ── Upload VTT ────────────────────────────────────────────────────────────
    print(f"\n📁  Upload your VTT file for this video:")
    uploaded_vtt = files.upload()
    vtt_name     = list(uploaded_vtt.keys())[0]
    vtt_raw      = uploaded_vtt[vtt_name].decode("utf-8", errors="replace")
    vtt_segments = parse_vtt(vtt_raw)
    print(f"  ✅ VTT parsed: {len(vtt_segments)} segments")

    # ── Type queries ──────────────────────────────────────────────────────────
    print(f"\n💬  Type your queries for this video.")
    print(f"    Press Enter after each one. Type DONE to finish.\n")
    queries = []
    while True:
        q = input(f"  Query {len(queries)+1}: ").strip()
        if q.upper() == "DONE" or q == "":
            break
        queries.append(q)

    if not queries:
        print("  ⚠  No queries entered — skipping this video.")
        video_count -= 1
        continue

    print(f"\n  ✅ {len(queries)} quer{'y' if len(queries)==1 else 'ies'} recorded")

    # ── Run all 8 configs ─────────────────────────────────────────────────────
    print(f"\n🔄  Running {len(CONFIGS)} configurations...\n")

    video_data = {
        "video_label":   video_label,
        "video_path":    video_path,
        "vtt_segments":  vtt_segments,
        "queries":       queries,
        "configs":       {},
    }

    for cfg_name, cfg in CONFIGS.items():
        print(f"  ▶  {cfg_name}")
        t0  = time.time()
        tag = f"v{video_count}_{cfg_name}"

        try:
            # Step 1 — transcription / VTT
            segments = step_transcribe(video_path, cfg, vtt_segments)

            # Step 2 — text index
            embed_model, text_embs, text_index, dim = step_text_index(segments, cfg)

            # Step 3 — frames
            frame_dir, fps = step_extract_frames(video_path, cfg, tag)

            # Step 4 — captions
            visual_segments = step_visual_captions(frame_dir, cfg)

            # Step 5 — visual index
            visual_embs, visual_index = step_visual_index(
                visual_segments, embed_model, dim)

            # Step 6 — answer every query
            query_outputs = []
            for qi, query in enumerate(queries):
                qtag = f"{tag}_q{qi+1}"
                out  = step_answer_query(
                    query, segments, text_index,
                    visual_segments, visual_index,
                    embed_model, cfg, video_path, qtag
                )
                # Compute automated metrics immediately
                rouge1, ar_auto   = auto_ar(out["answer"], "")  # no ref yet
                egs_rate, egs_auto = auto_egs(out["answer"], out["text_context"])
                tla_iou, tla_auto  = auto_tla(
                    out["clip_start"], out["clip_end"],
                    vtt_segments, query)

                query_outputs.append({
                    "query":       query,
                    "answer":      out["answer"],
                    "clip_path":   out["clip_path"],
                    "clip_dur_s":  out["clip_dur_s"],
                    "text_ctx":    out["text_context"],
                    "visual_ctx":  out["visual_context"],
                    "clip_start":  out["clip_start"],
                    "clip_end":    out["clip_end"],
                    # Automated scores
                    "rouge1":      rouge1,
                    "ar_auto":     ar_auto,
                    "egs_rate":    egs_rate,
                    "egs_auto":    egs_auto,
                    "tla_iou":     tla_iou,
                    "tla_auto":    tla_auto,
                    # Manual scores — filled in below
                    "ar_manual":   None,
                    "egs_manual":  None,
                    "tla_manual":  None,
                })

            elapsed = round(time.time() - t0, 1)
            video_data["configs"][cfg_name] = {
                "cfg":      cfg,
                "queries":  query_outputs,
                "time_s":   elapsed,
                "error":    None,
            }
            print(f"     ✅ Done in {elapsed}s\n")

        except Exception as e:
            elapsed = round(time.time() - t0, 1)
            video_data["configs"][cfg_name] = {
                "cfg":    cfg,
                "queries": [],
                "time_s":  elapsed,
                "error":   str(e),
            }
            print(f"     ❌ Error: {e}\n")

    # ── B2: Show all answers then score ───────────────────────────────────────
    print(f"\n{'═'*65}")
    print(f"  ANSWERS COLLECTED — Please score each one below")
    print(f"  Scoring scale: AR/EGS → 0=wrong  1=partial  2=correct")
    print(f"                 TLA    → 0=wrong clip  1=correct clip")
    print(f"{'═'*65}")

    for qi, query in enumerate(queries):
        print(f"\n  ── Query {qi+1}: {query}")
        print(f"  {'─'*55}")

        for cfg_name in CONFIGS:
            cdata = video_data["configs"].get(cfg_name, {})
            if cdata.get("error") or not cdata.get("queries"):
                continue
            qdata  = cdata["queries"][qi]
            answer = qdata["answer"]

            print(f"\n  [{cfg_name}]")
            print(f"  Answer    : {answer}")
            print(f"  Auto AR   : {qdata['ar_auto']}  "
                  f"Auto EGS : {qdata['egs_auto']}  "
                  f"Auto TLA : {qdata['tla_auto']}  "
                  f"(TLA-IoU={qdata['tla_iou']})")

        # Score after seeing all configs' answers for this query
        print(f"\n  ── Manual scores for Query {qi+1}: {query}")
        print(f"  (You will score each config separately)")

        for cfg_name in CONFIGS:
            cdata = video_data["configs"].get(cfg_name, {})
            if cdata.get("error") or not cdata.get("queries"):
                continue
            qdata = cdata["queries"][qi]
            print(f"\n    [{cfg_name}] Answer: {qdata['answer'][:80]}...")
            try:
                ar  = int(input(f"    AR  (0/1/2): ").strip())
                egs = int(input(f"    EGS (0/1/2): ").strip())
                tla = int(input(f"    TLA (0/1)  : ").strip())
                qdata["ar_manual"]  = max(0, min(2, ar))
                qdata["egs_manual"] = max(0, min(2, egs))
                qdata["tla_manual"] = max(0, min(1, tla))
            except ValueError:
                print("    Invalid input — scores set to 0")
                qdata["ar_manual"]  = 0
                qdata["egs_manual"] = 0
                qdata["tla_manual"] = 0

    # ── Compute aggregate metrics per config ──────────────────────────────────
    for cfg_name, cdata in video_data["configs"].items():
        if cdata["error"] or not cdata["queries"]:
            cdata["aggregate"] = None
            continue
        qs = cdata["queries"]
        N  = len(qs)

        def safe_avg(key):
            vals = [q[key] for q in qs if q[key] is not None]
            return round(sum(vals)/len(vals), 4) if vals else None

        cdata["aggregate"] = {
            # Automated
            "ar_auto_avg":   safe_avg("ar_auto"),
            "egs_auto_avg":  safe_avg("egs_auto"),
            "tla_auto_avg":  safe_avg("tla_auto"),
            "rouge1_avg":    safe_avg("rouge1"),
            "egs_rate_avg":  safe_avg("egs_rate"),
            "tla_iou_avg":   safe_avg("tla_iou"),
            # Manual (0-2 normalised to 0-1 for AR/EGS, TLA stays 0-1)
            "ar_manual_avg":  round(safe_avg("ar_manual")/2, 4)
                              if safe_avg("ar_manual") is not None else None,
            "egs_manual_avg": round(safe_avg("egs_manual")/2, 4)
                              if safe_avg("egs_manual") is not None else None,
            "tla_manual_avg": safe_avg("tla_manual"),
        }

    # Store
    vid_key = f"Video_{video_count:02d}_{video_label}"
    MASTER[vid_key] = video_data

    # Auto-save
    safe_master = {}
    for vk, vv in MASTER.items():
        safe_master[vk] = {
            k: v for k, v in vv.items() if k not in ("vtt_segments",)
        }
    with open("results/master_results.json", "w") as f:
        json.dump(safe_master, f, indent=2, default=str)
    print(f"\n  ✅ Results saved for {vid_key}")

    # ── Continue or stop ──────────────────────────────────────────────────────
    print(f"\n{'─'*65}")
    cont = input("  ▶  Upload another video? (y / n): ").strip().lower()
    if cont != "y":
        print(f"\n✅ All videos complete. Run Cell 5 to generate the report.")
        break

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5 ── GENERATE DOWNLOADABLE HTML REPORT
# Run this cell after finishing all videos in Cell 4
# ═══════════════════════════════════════════════════════════════════════════════

def _bar(value, max_val=1.0, color="#2196F3", width=80):
    """Render a small inline progress bar as HTML."""
    if value is None:
        return "<span style='color:#aaa'>—</span>"
    pct = min(100, round((value / max_val) * 100))
    return (f"<div style='display:flex;align-items:center;gap:6px'>"
            f"<div style='width:{width}px;height:8px;background:#eee;"
            f"border-radius:4px;overflow:hidden'>"
            f"<div style='width:{pct}%;height:100%;background:{color};"
            f"border-radius:4px'></div></div>"
            f"<span style='font-size:12px'>{value:.3f}</span></div>")

def _fmt(v, decimals=3):
    if v is None: return "—"
    return f"{v:.{decimals}f}"

def _score_color(v, thresholds=(0.4, 0.7)):
    if v is None: return "#888"
    if v >= thresholds[1]: return "#2e7d32"
    if v >= thresholds[0]: return "#e65100"
    return "#c62828"

def generate_report():
    now      = datetime.datetime.now()
    duration = str(now - SESSION_START).split(".")[0]

    # ── Table 1 HTML: Config parameter grid ──────────────────────────────────
    param_keys = [
        ("whisper_model_size",     "Whisper model"),
        ("whisper_temperature",    "Whisper temp"),
        ("whisper_beam_size",      "Whisper beams"),
        ("blip_model_size",        "BLIP model"),
        ("blip_num_beams",         "BLIP beams"),
        ("blip_min_new_tokens",    "BLIP min tokens"),
        ("blip_max_new_tokens",    "BLIP max tokens"),
        ("embed_model",            "Embed model"),
        ("top_k",                  "Top-k"),
        ("frame_interval_s",       "Frame interval"),
        ("flan_num_beams",         "FLAN beams"),
        ("flan_temperature",       "FLAN temp"),
        ("flan_repetition_penalty","FLAN rep penalty"),
        ("flan_length_penalty",    "FLAN len penalty"),
        ("flan_max_length",        "FLAN max len"),
    ]

    cfg1 = CONFIGS["Config_1_Baseline"]  # reference for highlighting changes

    t1_header = "".join(
        f"<th>{label}</th>" for _, label in param_keys
    )
    t1_rows = ""
    for cfg_name, cfg in CONFIGS.items():
        cells = ""
        for key, _ in param_keys:
            val     = cfg.get(key, "—")
            ref_val = cfg1.get(key, "—")
            # Shorten BLIP model name
            display_val = str(val).replace("Salesforce/blip-image-captioning-","blip-")
            changed = (val != ref_val and cfg_name != "Config_1_Baseline")
            bg      = "background:#fff9c4;" if changed else ""
            cells  += f"<td style='{bg}font-size:12px'>{display_val}</td>"
        hyp = cfg.get("hypothesis","")
        t1_rows += (f"<tr><td style='font-weight:500;white-space:nowrap'>"
                    f"{cfg_name}</td>"
                    f"<td style='font-size:11px;color:#555;max-width:200px'>"
                    f"{hyp}</td>"
                    f"{cells}</tr>")

    table1_html = f"""
    <h2>Table 1 — Parameter Configurations</h2>
    <p style='font-size:13px;color:#555;margin-bottom:10px'>
      Yellow cells indicate parameters changed from Config_1 baseline.
      Config_8 combines the best settings from all individual configs.
    </p>
    <div style='overflow-x:auto'>
    <table>
      <thead><tr>
        <th>Config</th><th>Hypothesis</th>{t1_header}
      </tr></thead>
      <tbody>{t1_rows}</tbody>
    </table>
    </div>"""

    # ── Table 2 HTML: Per-video per-config metrics ────────────────────────────
    table2_html = "<h2>Table 2 — Performance Metrics per Video per Configuration</h2>"
    table2_html += """
    <p style='font-size:13px;color:#555;margin-bottom:10px'>
      AR = Answer Relevance &nbsp;|&nbsp;
      EGS = Evidence Grounding Score &nbsp;|&nbsp;
      TLA = Temporal Localization Accuracy<br>
      <b>Auto</b> = computed automatically &nbsp;|&nbsp;
      <b>Manual</b> = your 0/1/2 scores normalised to [0,1]
    </p>"""

    for vid_key, vdata in MASTER.items():
        table2_html += f"<h3 style='margin-top:24px'>📹 {vid_key}</h3>"
        table2_html += "<div style='overflow-x:auto'><table>"
        table2_html += """<thead><tr>
          <th>Config</th>
          <th>AR (auto)</th><th>AR (manual)</th>
          <th>EGS (auto)</th><th>EGS (manual)</th>
          <th>TLA IoU</th><th>TLA (manual)</th>
          <th>ROUGE-1</th><th>Grounding rate</th>
          <th>Build time</th>
        </tr></thead><tbody>"""

        # Find best combined score for highlighting
        best_score, best_cfg = -1, None
        for cfg_name, cdata in vdata["configs"].items():
            if cdata.get("aggregate"):
                agg = cdata["aggregate"]
                vals = [agg.get("ar_manual_avg"), agg.get("egs_manual_avg"),
                        agg.get("tla_manual_avg")]
                vals = [v for v in vals if v is not None]
                score = sum(vals)/len(vals) if vals else 0
                if score > best_score:
                    best_score, best_cfg = score, cfg_name

        for cfg_name, cdata in vdata["configs"].items():
            if cdata.get("error"):
                table2_html += (f"<tr><td>{cfg_name}</td>"
                                f"<td colspan='9' style='color:#c62828'>"
                                f"Error: {cdata['error']}</td></tr>")
                continue
            agg = cdata.get("aggregate")
            if not agg:
                continue
            is_best = (cfg_name == best_cfg)
            row_bg  = "background:#e8f5e9;" if is_best else ""
            medal   = " 🏆" if is_best else ""

            table2_html += f"<tr style='{row_bg}'>"
            table2_html += (f"<td style='font-weight:500;white-space:nowrap'>"
                            f"{cfg_name}{medal}</td>")

            cols = [
                (agg.get("ar_auto_avg"),   "#2196F3"),
                (agg.get("ar_manual_avg"), "#1565C0"),
                (agg.get("egs_auto_avg"),  "#4CAF50"),
                (agg.get("egs_manual_avg"),"#2e7d32"),
                (agg.get("tla_iou_avg"),   "#FF9800"),
                (agg.get("tla_manual_avg"),"#e65100"),
                (agg.get("rouge1_avg"),    "#9C27B0"),
                (agg.get("egs_rate_avg"),  "#00897B"),
            ]
            for val, color in cols:
                table2_html += f"<td>{_bar(val, color=color)}</td>"

            table2_html += (f"<td style='font-size:12px'>"
                            f"{cdata['time_s']}s</td></tr>")

        table2_html += "</tbody></table></div>"

    # Per-query detail tables
    table2_html += "<h3 style='margin-top:32px'>Per-Query Detail</h3>"
    for vid_key, vdata in MASTER.items():
        table2_html += f"<h4 style='margin-top:16px'>📹 {vid_key}</h4>"
        for qi, query in enumerate(vdata["queries"]):
            table2_html += (f"<p style='font-size:13px;font-weight:500;"
                            f"margin:12px 0 6px'>Q{qi+1}: {query}</p>")
            table2_html += "<div style='overflow-x:auto'><table>"
            table2_html += """<thead><tr>
              <th>Config</th><th>Generated Answer</th>
              <th>AR auto</th><th>AR manual</th>
              <th>EGS auto</th><th>EGS manual</th>
              <th>TLA IoU</th><th>TLA manual</th>
              <th>Clip (s)</th>
            </tr></thead><tbody>"""

            for cfg_name, cdata in vdata["configs"].items():
                if cdata.get("error") or not cdata.get("queries"):
                    continue
                if qi >= len(cdata["queries"]):
                    continue
                qd  = cdata["queries"][qi]
                table2_html += "<tr>"
                table2_html += (f"<td style='font-weight:500;white-space:nowrap;"
                                f"font-size:12px'>{cfg_name}</td>")
                table2_html += (f"<td style='font-size:12px;max-width:220px'>"
                                f"{qd['answer']}</td>")

                for val, color in [
                    (qd.get("ar_auto"),  "#2196F3"),
                    (qd.get("ar_manual"),"#1565C0"),
                    (qd.get("egs_auto"), "#4CAF50"),
                    (qd.get("egs_manual"),"#2e7d32"),
                    (qd.get("tla_iou"),  "#FF9800"),
                    (qd.get("tla_manual"),"#e65100"),
                ]:
                    disp = _fmt(val/2 if val is not None and color in
                                ("#1565C0","#2e7d32") else val)
                    c    = _score_color(
                        val/2 if val is not None and color in
                        ("#1565C0","#2e7d32") else val)
                    table2_html += (f"<td style='font-size:12px;font-weight:500;"
                                    f"color:{c}'>{disp}</td>")

                table2_html += (f"<td style='font-size:12px'>"
                                f"{qd.get('clip_dur_s','—')}s</td>")
                table2_html += "</tr>"

            table2_html += "</tbody></table></div>"

    # ── Table 3 HTML: Best config per video + overall ─────────────────────────
    t3_rows = ""
    overall_scores = {cfg_name: [] for cfg_name in CONFIGS}

    for vid_key, vdata in MASTER.items():
        best_name, best_agg, best_score = None, None, -1
        for cfg_name, cdata in vdata["configs"].items():
            agg = cdata.get("aggregate")
            if not agg:
                continue
            vals  = [agg.get("ar_manual_avg"), agg.get("egs_manual_avg"),
                     agg.get("tla_manual_avg")]
            vals  = [v for v in vals if v is not None]
            score = sum(vals)/len(vals) if vals else 0
            overall_scores[cfg_name].append(score)
            if score > best_score:
                best_score, best_name, best_agg = score, cfg_name, agg

        if best_agg:
            t3_rows += (
                f"<tr>"
                f"<td style='font-weight:500'>{vid_key}</td>"
                f"<td style='font-weight:500;color:#1565C0'>{best_name}</td>"
                f"<td style='color:#2e7d32;font-weight:500'>{best_score:.3f}</td>"
                f"<td>{_fmt(best_agg.get('ar_manual_avg'))}</td>"
                f"<td>{_fmt(best_agg.get('egs_manual_avg'))}</td>"
                f"<td>{_fmt(best_agg.get('tla_manual_avg'))}</td>"
                f"<td>{_fmt(best_agg.get('rouge1_avg'))}</td>"
                f"</tr>"
            )

    # Overall best config
    avg_overall = {k: round(sum(v)/len(v),4) if v else 0
                   for k, v in overall_scores.items()}
    overall_best_name  = max(avg_overall, key=avg_overall.get)
    overall_best_score = avg_overall[overall_best_name]

    table3_html = f"""
    <h2>Table 3 — Best Configuration per Video &amp; Overall Best</h2>
    <div style='background:#e8f5e9;border:1px solid #a5d6a7;border-radius:8px;
                padding:16px;margin-bottom:16px'>
      <p style='font-size:15px;font-weight:600;color:#1b5e20;margin:0 0 8px'>
        🏆 Overall Best Configuration: {overall_best_name}
      </p>
      <p style='font-size:13px;color:#2e7d32;margin:0'>
        Average combined score across all videos: {overall_best_score:.4f}
      </p>
      <p style='font-size:13px;color:#555;margin:6px 0 0'>
        {CONFIGS[overall_best_name]["hypothesis"]}
      </p>
    </div>
    <div style='overflow-x:auto'>
    <table>
      <thead><tr>
        <th>Video</th><th>Best Config</th><th>Combined Score</th>
        <th>AR (manual)</th><th>EGS (manual)</th>
        <th>TLA (manual)</th><th>ROUGE-1</th>
      </tr></thead>
      <tbody>{t3_rows}</tbody>
    </table>
    </div>"""

    # ── Metric definitions footnote ───────────────────────────────────────────
    footnote_html = """
    <h2>Metric Definitions (from your paper)</h2>
    <table>
      <thead><tr><th>Metric</th><th>Definition</th><th>Formula</th>
      <th>Scale</th><th>How computed here</th></tr></thead>
      <tbody>
        <tr>
          <td><b>AR</b><br>Answer Relevance</td>
          <td>Whether the generated answer correctly addresses the user query</td>
          <td>AR = Σ sᵢ / 2N</td>
          <td>0–1 (normalised from 0/1/2)</td>
          <td>Auto: ROUGE-1 mapped to 0/1/2<br>Manual: you scored 0/1/2</td>
        </tr>
        <tr>
          <td><b>EGS</b><br>Evidence Grounding Score</td>
          <td>Whether the answer is explicitly supported by retrieved evidence</td>
          <td>EGS = Σ gᵢ / 2N</td>
          <td>0–1 (normalised from 0/1/2)</td>
          <td>Auto: content-word overlap with context<br>Manual: you scored 0/1/2</td>
        </tr>
        <tr>
          <td><b>TLA</b><br>Temporal Localization Accuracy</td>
          <td>Whether the extracted clip corresponds to the correct timestamp</td>
          <td>TLA = Correct clips / N</td>
          <td>0–1 (binary)</td>
          <td>Auto: IoU ≥ 0.5 with VTT timestamp<br>Manual: you scored 0/1</td>
        </tr>
      </tbody>
    </table>"""

    # ── Assemble full HTML ────────────────────────────────────────────────────
    total_queries = sum(
        len(v["queries"]) for v in MASTER.values()
    )
    total_runs = sum(
        len(v["configs"]) for v in MASTER.values()
    )

    html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>SummariV — Hyperparameter Tuning Report</title>
<style>
  *{{box-sizing:border-box;margin:0;padding:0}}
  body{{font-family:'Segoe UI',Arial,sans-serif;background:#f5f7fa;
        color:#1a1a1a;font-size:14px;line-height:1.6}}
  .header{{background:linear-gradient(135deg,#1F4E79,#2E75B6);
           color:#fff;padding:40px 60px}}
  .header h1{{font-size:2em;margin-bottom:8px}}
  .header p{{opacity:.85;font-size:13px}}
  .meta{{display:grid;grid-template-columns:repeat(4,1fr);
         gap:16px;padding:28px 60px}}
  .meta-card{{background:#fff;border-radius:10px;padding:18px;
              box-shadow:0 2px 8px rgba(0,0,0,.07);
              border-left:4px solid #2E75B6;text-align:center}}
  .meta-card .num{{font-size:2em;font-weight:600;color:#1F4E79}}
  .meta-card .lbl{{font-size:12px;color:#666;margin-top:4px}}
  .section{{background:#fff;border-radius:10px;margin:16px 60px;
            padding:28px;box-shadow:0 2px 8px rgba(0,0,0,.06)}}
  h2{{font-size:1.3em;color:#1F4E79;margin-bottom:14px;
      padding-bottom:8px;border-bottom:2px solid #e3eef9}}
  h3{{font-size:1.05em;color:#2E75B6;margin:16px 0 8px}}
  h4{{font-size:.95em;color:#444;margin:12px 0 6px}}
  table{{width:100%;border-collapse:collapse;font-size:13px}}
  thead tr{{background:#1F4E79;color:#fff}}
  th{{padding:10px 12px;text-align:left;white-space:nowrap;font-weight:500}}
  td{{padding:8px 12px;border-bottom:1px solid #eee;vertical-align:middle}}
  tr:hover{{background:#f0f7ff!important}}
  .footer{{text-align:center;padding:28px;color:#999;font-size:12px}}
</style>
</head>
<body>

<div class="header">
  <h1>SummariV — Hyperparameter Tuning Report</h1>
  <p>Generated: {now:%Y-%m-%d %H:%M:%S} &nbsp;|&nbsp; Session duration: {duration}</p>
  <p>Multimodal Retrieval-Augmented Video Question Answering</p>
</div>

<div class="meta">
  <div class="meta-card">
    <div class="num">{len(MASTER)}</div>
    <div class="lbl">Videos evaluated</div>
  </div>
  <div class="meta-card">
    <div class="num">{len(CONFIGS)}</div>
    <div class="lbl">Configurations</div>
  </div>
  <div class="meta-card">
    <div class="num">{total_queries}</div>
    <div class="lbl">Total queries</div>
  </div>
  <div class="meta-card">
    <div class="num">{total_runs}</div>
    <div class="lbl">Pipeline runs</div>
  </div>
</div>

<div class="section">{table1_html}</div>
<div class="section">{table2_html}</div>
<div class="section">{table3_html}</div>
<div class="section">{footnote_html}</div>

<div class="footer">
  SummariV Hyperparameter Tuning Report &nbsp;|&nbsp;
  {now:%Y-%m-%d} &nbsp;|&nbsp;
  {len(CONFIGS)} configs × {len(MASTER)} videos
</div>

</body>
</html>"""

    # Save and download
    report_path = "results/SummariV_HyperTuning_Report.html"
    with open(report_path, "w", encoding="utf-8") as f:
        f.write(html)

    print(f"✅ Report saved: {report_path}")
    print("   Downloading now...")
    files.download(report_path)
    display(HTML(f"""
    <div style='background:#e8f5e9;border:2px solid #4CAF50;border-radius:10px;
                padding:16px;font-family:Arial;margin:12px 0'>
      <b style='color:#2e7d32'>✅ Report downloaded</b><br>
      <span style='font-size:13px;color:#555'>
        Open <b>SummariV_HyperTuning_Report.html</b> in your browser to view.
      </span>
    </div>"""))


# Run the report
generate_report()